In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)


In [ ]:
# Task 2: Write your code here:

print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3: Write your code here:
# Check data types and structure
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food = df_food.drop(columns=['Order_ID']).copy()
print(f"Shape after cleaning: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df_food.isnull().sum())

In [ ]:
# Select relevant columns,
cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()

# Drop rows where target or key features are missing - can't predict without them
print(f"Before: {df_food.shape}")
df_clean = df_clean.dropna(subset=['Time_of_Day', 'Traffic_Level', 'Delivery_Time' ,'Courier_Experience_yrs'])
print(f"After dropping missing value: {df_clean.shape}")

In [ ]:
# Fill categorical columns with 'unknown' since  can predict without it
df_clean['Weather'] = df_clean['Weather'].fillna('unknown')

In [ ]:
# Task 3: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# Do we have categorical columns?
categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Encode Categorical Features
label_encoder = LabelEncoder()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = label_encoder.fit_transform(df_clean[col])

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', df_clean) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df_clean) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")
#Dataset  imbalance

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)




In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []


for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Gather importances from the models (from the last fold)

importances = {}

importances['Random Forest'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(y_fold_pred, bins=50, edgecolor='black', color='green')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: